In [ ]:
import pandas as pd
import numpy as np
import json
import re
import time
import os
from dotenv import load_dotenv

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import openai

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import statsmodels.api as sm
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df_post = pd.read_pickle("../data/oon/df_post_oon.pkl")

In [ ]:
# Load environment variables from a .env file
load_dotenv()

# Get the API key
api_key = os.getenv("DLAB_API_KEY")
if not api_key:
    raise ValueError("DLAB_API_KEY not found in .env file or environment variables")

In [ ]:
emotions = [
    'anger', 'fear', 'sadness', 'disgust'
]

In [ ]:
client = openai.OpenAI(api_key=api_key)

[Tested several prompts and did qualitative check with post samples here (retrieved)]

In [ ]:
def get_llm_emotion_detection(text):
    prompt = f"""
You are an emotionally-intelligent and empathetic agent.
You will be given a piece of text derived from posts in the subreddit r/QAnonCasualties, which are personal narratives describing one's experiences with a loved one who has been radicalized.
You must identify the emotions explicitly stated or clearly implied by the writer about their own feelings.
Only consider the writer’s emotions — ignore the emotions of other people mentioned.
If the emotion is expressed at least once in the text, output 1; otherwise, output 0.
Do not infer emotions based solely on the topic or context unless the writer states or clearly implies them.
Possible emotions: Anger, Disgust, Sadness, Fear, Surprise.

Text: "{text}"

Please provide outputs for these emotions in JSON format:
{{
    "anger": 0 or 1,
    "fear": 0 or 1,
    "sadness": 0 or 1,
    "disgust": 0 or 1,
}}

Only respond with the JSON object, no additional text.
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini-2024-07-18",
            temperature=0.2,
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response.choices[0].message.content.strip()
        answer = re.sub(r'^```(?:json)?|```$', '', answer.strip(), flags=re.MULTILINE).strip()
        return json.loads(answer)
        
    except Exception as e:
        print(f"Error: {e}")
        return {emotion: 0.0 for emotion in basic_emotions}

In [ ]:
# Now apply (the second prompt) to the whole set
for idx, row in tqdm(df_post.iterrows()):
    emo_scores = get_llm_emotion_detection(row['selftext'])

    for emo in emotions:
        df_post.at[idx, f'emotion__{emo}'] = emo_scores[emo]

In [ ]:
df_post[['emotion__anger', 'emotion__fear', 'emotion__sadness', 'emotion__disgust', 'emotion__surprise']].describe()

,emotion__anger,emotion__fear,emotion__sadness,emotion__disgust,emotion__surprise
count,10860.000000,10860.000000,10860.000000,10860.000000,10860.000000
mean,0.659116,0.562615,0.826611,0.553591,0.240792
std,0.474028,0.496087,0.378600,0.497143,0.427584
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,0.000000,0.000000
50%,1.000000,1.000000,1.000000,1.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
df_post.to_pickle("../data/oon/df_post_oon.pkl")